# Is it useful ?!

In [15]:
import os
import pandas as pd

DATA_DIR = "/nfs/scratch/pdb_dimers/KaHIP"

In [16]:
output = pd.read_csv(os.path.join(DATA_DIR, "output.txt"), header=None)
output.head()

,0
0,3
1,3
2,3
3,3
4,3


In [17]:
output["cluster_id"] = output.index.map(lambda x: x + 1)  # METIS uses 1-based indexing
output.head()

,0,cluster_id
0,3,1
1,3,2
2,3,3
3,3,4
4,3,5


In [18]:
# Plot the distribution of cluster sizes
cluster_sizes = output[0].value_counts().sort_index()
print(cluster_sizes)

0
0    799
1    799
2    798
3    796
4    798
5    799
6    798
7    798
8    799
9    799
Name: count, dtype: int64


# Partitioning with sequence identity

In [19]:
seq_ident_output = pd.read_csv(os.path.join(DATA_DIR, "seq_ident_partitions_strong_output.txt"), header=None)
seq_ident_output.head()

,0
0,2
1,8
2,4
3,2
4,7


In [20]:
len(seq_ident_output)

38250

In [21]:
seq_ident_cluster_sizes = seq_ident_output[0].value_counts().sort_index()
print(seq_ident_cluster_sizes)

0
0    3938
1    3938
2    3938
3    2808
4    3938
5    3938
6    3938
7    3938
8    3938
9    3938
Name: count, dtype: int64


# Fuse partition plus into interaction df

In [22]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions.tsv", sep="\t")
partitions = pd.read_csv("/nfs/scratch/pdb_dimers/KaHIP/seq_ident_partitions_strong_output.txt", header=None)

partitions.head()

,0
0,2
1,8
2,4
3,2
4,7


In [23]:
len(partitions)

38250

In [24]:
partitions = partitions[0]

int_df["partitions"] = partitions

int_df.head()

,assembly_id,pdb_id,assembly_number,entity_pair,uniprot_pair,uniprot_1,uniprot_2,species_pair,species_1,species_2,...,dimer_type,cluster_pair_100pct,new_cluster_pair,resolution_best_angstrom,modeled_polymer_monomer_count,experimental_method,oligomeric_count,download_url,local_filename,partitions
0,10BL-1,10BL,1,"10BL_1,10BL_1",Q4E2L0|Q4E2L0,Q4E2L0,Q4E2L0,Trypanosoma cruzi|Trypanosoma cruzi,Trypanosoma cruzi,Trypanosoma cruzi,...,homo,"20385_100,20385_100","fix_18845_100,fix_18845_100",2.60,664,X-ray,2,https://files.rcsb.org/download/10bl-assembly1...,10bl-assembly1.cif.gz,2
1,10FT-1,10FT,1,"10FT_1,10FT_2",Q99435|Q78DX7,Q99435,Q78DX7,Homo sapiens|Mus musculus,Homo sapiens,Mus musculus,...,hetero,"10609_100,17054_100","fix_14530_100,fix_37352_100",3.21,775,EM,2,https://files.rcsb.org/download/10ft-assembly1...,10ft-assembly1.cif.gz,8
2,10GS-1,10GS,1,"10GS_1,10GS_1",P09211|P09211,P09211,P09211,Homo sapiens|Homo sapiens,Homo sapiens,Homo sapiens,...,homo,"1221_100,1221_100","fix_34000_100,fix_34000_100",2.20,416,X-ray,2,https://files.rcsb.org/download/10gs-assembly1...,10gs-assembly1.cif.gz,4
3,10JU-1,10JU,1,"10JU_1,10JU_2",Q582V7|Q582V7,Q582V7,Q582V7,Trypanosoma brucei brucei TREU927|Trypanosoma ...,Trypanosoma brucei brucei TREU927,Trypanosoma brucei brucei TREU927,...,hetero,"65494_100,65494_100","fix_18848_100,fix_18846_100",2.15,522,X-ray,2,https://files.rcsb.org/download/10ju-assembly1...,10ju-assembly1.cif.gz,2
4,10JX-1,10JX,1,"10JX_1,10JX_1",A0A068NTE8|A0A068NTE8,A0A068NTE8,A0A068NTE8,Fimbriimonas ginsengisoli Gsoil 348|Fimbriimon...,Fimbriimonas ginsengisoli Gsoil 348,Fimbriimonas ginsengisoli Gsoil 348,...,homo,"65388_100,65388_100","fix_6249_100,fix_6249_100",1.47,282,X-ray,2,https://files.rcsb.org/download/10jx-assembly1...,10jx-assembly1.cif.gz,7


In [25]:
all_homo = 0
all_hetero = 0
for i in range(10):
    int_df_tmp = int_df[int_df["partitions"] == i]
    counts = int_df_tmp["dimer_type"].value_counts(normalize=True)*100
    all_homo += counts["homo"]
    all_hetero += counts["hetero"]
    print(f"{i}:\thomo: {counts["homo"]}\thetero: {counts["hetero"]}")

print("\n")
print(f"Avg Counts:\nhomo: {all_homo/10}\thetero: {all_hetero/10}")

0:	homo: 31.10716099542915	hetero: 68.89283900457085
1:	homo: 60.43676993397664	hetero: 39.563230066023365
2:	homo: 84.45911630269171	hetero: 15.540883697308278
3:	homo: 90.38461538461539	hetero: 9.615384615384617
4:	homo: 81.4118842051803	hetero: 18.588115794819707
5:	homo: 82.5038090401219	hetero: 17.49619095987811
6:	homo: 87.07465718638903	hetero: 12.925342813610971
7:	homo: 91.21381411884205	hetero: 8.786185881157948
8:	homo: 89.00457084814627	hetero: 10.995429151853733
9:	homo: 95.02285424073133	hetero: 4.977145759268664


Avg Counts:
homo: 79.26192522561237	hetero: 20.738074774387623


In [26]:
int_df["split"] = int_df["partitions"].map({
    0: "train",
    1: "train",
    2: "train",
    3: "train",
    4: "test",
    5: "val",
    6: "train",
    7: "train",
    8: "train",
    9: "train",
})

In [27]:
int_df.to_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions_with_partitions.tsv", sep="\t")

# homo vs hetero distribution in different splits

In [11]:
int_df = pd.read_csv("/nfs/scratch/pdb_dimers/final_filtered_interactions_with_partitions.tsv", sep="\t")


int_df_train = int_df[int_df["split"]=="train"]
int_df_val = int_df[int_df["split"]=="val"]
int_df_test = int_df[int_df["split"]=="test"]

len(int_df_test)

2275

In [23]:
def calculate_percentages(df):
    return df["dimer_type"].value_counts(normalize=True) * 100


calculate_percentages(int_df_train)

dimer_type
homo      78.874765
hetero    21.125235
Name: proportion, dtype: float64

In [24]:
calculate_percentages(int_df_val)

dimer_type
homo      83.648352
hetero    16.351648
Name: proportion, dtype: float64

In [25]:
calculate_percentages(int_df_test)

dimer_type
homo      82.065934
hetero    17.934066
Name: proportion, dtype: float64